# Komparasi Kinerja Model XGBoost vs CatBoost dengan SMOTE

Analisis perbandingan performa algoritma **XGBoost** dan **CatBoost** untuk klasifikasi data tidak seimbang menggunakan teknik **SMOTE** (Synthetic Minority Over-sampling Technique).

## Dataset
**Bank Marketing Dataset** - Prediksi apakah nasabah akan berlangganan deposito berjangka atau tidak.

---

## 1. Exploratory Data Analysis (EDA)

In [1]:
# Import library yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc
import xgboost as xgb
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [2]:
# Load dataset
df = pd.read_csv("bank-additional-full.csv", sep=';')

# Tampilkan informasi dasar dataset
print("=" * 60)
print("INFORMASI DATASET")
print("=" * 60)
print(f"\nJumlah baris dan kolom: {df.shape}")
print(f"\nTipe data kolom:")
print(df.dtypes)
print(f"\nSample data (5 baris pertama):")
print(df.head())
print(f"\nInformasi missing values:")
print(df.isnull().sum())
print(f"\nStatistik deskriptif:")
print(df.describe())

INFORMASI DATASET

Jumlah baris dan kolom: (41188, 21)

Tipe data kolom:
age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object

Sample data (5 baris pertama):
   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     

In [ ]:
# Analisis distribusi target variable
print("=" * 60)
print("ANALISIS TARGET VARIABLE (y)")
print("=" * 60)

print(f"\nDistribusi kelas target:")
print(df['y'].value_counts())
print(f"\nProporsi kelas:")
print(df['y'].value_counts(normalize=True))

# Visualisasi distribusi kelas
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
df['y'].value_counts().plot(kind='bar', color=['#FF6B6B', '#4ECDC4'])
plt.title('Distribusi Kelas Target', fontsize=14, fontweight='bold')
plt.xlabel('Kelas')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
df['y'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4'])
plt.title('Proporsi Kelas Target', fontsize=14, fontweight='bold')
plt.ylabel('')

plt.tight_layout()
plt.show()

# Hitung rasio ketidakseimbangan
no_count = df['y'].value_counts()['no']
yes_count = df['y'].value_counts()['yes']
imbalance_ratio = no_count / yes_count
print(f"\n⚠️ Rasio Ketidakseimbangan (No:Yes) = {imbalance_ratio:.2f}:1")
print(f"   Dataset sangat tidak seimbang! Kelas 'no' {imbalance_ratio:.2f}x lebih banyak dari 'yes'")

## 2. Preprocessing Data

In [ ]:
# Pemisahan fitur (X) dan target (y)
X = df.drop('y', axis=1)
y = df['y'].apply(lambda x: 1 if x == 'yes' else 0)  # Konversi ke binary (0: no, 1: yes)

print("=" * 60)
print("PEMISAHAN FITUR DAN TARGET")
print("=" * 60)
print(f"\nJumlah fitur: {X.shape[1]}")
print(f"Jumlah sampel: {X.shape[0]}")
print(f"\nDistribusi target setelah encoding:")
print(y.value_counts())

In [ ]:
# Identifikasi tipe fitur (kategorikal vs numerik)
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("=" * 60)
print("IDENTIFIKASI TIPE FITUR")
print("=" * 60)
print(f"\nFitur Kategorikal ({len(categorical_features)}):")
for i, col in enumerate(categorical_features, 1):
    print(f"  {i}. {col} - {X[col].nunique()} kategori unik")

print(f"\nFitur Numerik ({len(numeric_features)}):")
for i, col in enumerate(numeric_features, 1):
    print(f"  {i}. {col}")

In [ ]:
# Split data menjadi training dan testing set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% untuk testing
    random_state=42,    # Untuk reproduksibilitas
    stratify=y          # Menjaga proporsi kelas di train dan test
)

print("=" * 60)
print("PEMBAGIAN DATA (TRAIN-TEST SPLIT)")
print("=" * 60)
print(f"\nData Training:")
print(f"  - Jumlah sampel: {X_train.shape[0]}")
print(f"  - Distribusi kelas: {y_train.value_counts().to_dict()}")

print(f"\nData Testing:")
print(f"  - Jumlah sampel: {X_test.shape[0]}")
print(f"  - Distribusi kelas: {y_test.value_counts().to_dict()}")

print(f"\nRasio Train:Test = {len(X_train)}:{len(X_test)} ≈ 80:20")

## 3. Implementasi Model

### 3.1 XGBoost dengan SMOTE

In [ ]:
# Preprocessing untuk XGBoost: One-Hot Encoding (OHE) untuk fitur kategorikal
preprocessor_xgb = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'  # Fitur numerik dibiarkan apa adanya
)

# Inisialisasi model XGBoost dengan hyperparameter yang sudah di-tuning
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',  # Klasifikasi binary
    eval_metric='logloss',        # Metrik evaluasi
    random_state=42,
    n_estimators=1000,            # Jumlah pohon keputusan
    learning_rate=0.05,           # Laju pembelajaran
    max_depth=5,                  # Kedalaman maksimum pohon
    subsample=0.7,                # Proporsi sampel untuk setiap pohon
    colsample_bytree=0.7,         # Proporsi fitur untuk setiap pohon
    n_jobs=-1                     # Gunakan semua CPU core
)

# Buat pipeline: Preprocessing → SMOTE → XGBoost
pipeline_xgb = ImbPipeline(steps=[
    ('preprocessor', preprocessor_xgb),  # Step 1: One-Hot Encoding
    ('smote', SMOTE(random_state=42)),   # Step 2: Oversampling dengan SMOTE
    ('classifier', xgb_model)            # Step 3: Training XGBoost
])

print("=" * 60)
print("TRAINING MODEL XGBOOST")
print("=" * 60)
print("\n🔄 Training sedang berjalan...")
print("   Pipeline: OHE → SMOTE → XGBoost")

# Training model
pipeline_xgb.fit(X_train, y_train)

# Prediksi pada data testing
y_pred_xgb = pipeline_xgb.predict(X_test)
y_proba_xgb = pipeline_xgb.predict_proba(X_test)[:, 1]

print("✅ Training XGBoost selesai!")

### 3.2 CatBoost dengan SMOTE

In [ ]:
# Pisahkan fitur numerik dan kategorikal dari data training
X_train_num = X_train[numeric_features]
X_train_cat = X_train[categorical_features]

# One-Hot Encoding untuk fitur kategorikal (diperlukan untuk SMOTE)
encoder_temp = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat_encoded = encoder_temp.fit_transform(X_train_cat)
cat_encoded_names = encoder_temp.get_feature_names_out(categorical_features)

# Gabungkan fitur numerik dan kategorikal yang sudah di-encode
X_train_processed = pd.DataFrame(
    np.hstack([X_train_num.values, X_train_cat_encoded]),
    columns=numeric_features + list(cat_encoded_names)
)

# Terapkan SMOTE untuk menangani data tidak seimbang
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_processed, y_train)

print("=" * 60)
print("PREPROCESSING UNTUK CATBOOST")
print("=" * 60)
print(f"\nSebelum SMOTE:")
print(f"  - Kelas 0 (no): {y_train.value_counts()[0]} sampel")
print(f"  - Kelas 1 (yes): {y_train.value_counts()[1]} sampel")

print(f"\nSetelah SMOTE:")
print(f"  - Kelas 0 (no): {y_smote.value_counts()[0]} sampel")
print(f"  - Kelas 1 (yes): {y_smote.value_counts()[1]} sampel")
print(f"\n✅ Data sekarang seimbang (balanced)!")

In [ ]:
# Inisialisasi model CatBoost
# CATATAN: cat_features dihapus karena data sudah di-encode menjadi numerik
cat_model = CatBoostClassifier(
    verbose=0,              # Tidak menampilkan log training
    random_state=42,
    iterations=500,         # Jumlah iterasi
    learning_rate=0.08,     # Laju pembelajaran
    depth=6,                # Kedalaman pohon
    l2_leaf_reg=5          # Regularisasi L2
)

print("=" * 60)
print("TRAINING MODEL CATBOOST")
print("=" * 60)
print("\n🔄 Training sedang berjalan...")
print("   Pipeline: OHE → SMOTE → CatBoost")

# Training model pada data yang sudah di-SMOTE
cat_model.fit(X_smote, y_smote)

# Transformasi data testing menggunakan preprocessor yang sama
X_test_processed = preprocessor_xgb.transform(X_test)

# Prediksi pada data testing
y_pred_cat = cat_model.predict(X_test_processed).flatten()
y_proba_cat = cat_model.predict_proba(X_test_processed)[:, 1]

print("✅ Training CatBoost selesai!")

## 4. Evaluasi Model

In [ ]:
# Fungsi untuk menghitung metrik evaluasi
def get_metrics(y_true, y_pred, y_proba, model_name):
    """
    Menghitung berbagai metrik evaluasi untuk klasifikasi binary
    
    Parameters:
    -----------
    y_true : array-like
        Label sebenarnya
    y_pred : array-like
        Prediksi kelas
    y_proba : array-like
        Probabilitas kelas positif
    model_name : str
        Nama model
    
    Returns:
    --------
    dict : Dictionary berisi metrik evaluasi
    """
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'AUC-ROC': roc_auc_score(y_true, y_proba)
    }

# Hitung metrik untuk kedua model
metrics_xgb = get_metrics(y_test, y_pred_xgb, y_proba_xgb, 'XGBoost (SMOTE)')
metrics_cat = get_metrics(y_test, y_pred_cat, y_proba_cat, 'CatBoost (SMOTE+OHE)')

# Buat DataFrame untuk perbandingan
results_df = pd.DataFrame([metrics_xgb, metrics_cat]).set_index('Model')

# Tampilkan hasil
print("=" * 80)
print("HASIL EVALUASI MODEL")
print("=" * 80)
print("\nPerbandingan Kinerja XGBoost vs CatBoost:\n")
print(results_df.T.round(4))
print("\n" + "=" * 80)
print("\nTabel Markdown:")
print(results_df.T.round(4).to_markdown(numalign="left", stralign="left"))

In [ ]:
# Visualisasi: Kurva ROC
print("=" * 60)
print("VISUALISASI KURVA ROC")
print("=" * 60)

# Hitung TPR dan FPR untuk kedua model
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)
roc_auc_xgb = auc(fpr_xgb, tpr_xgb)

fpr_cat, tpr_cat, _ = roc_curve(y_test, y_proba_cat)
roc_auc_cat = auc(fpr_cat, tpr_cat)

# Plotting
plt.figure(figsize=(10, 7))

# Plot CatBoost
plt.plot(
    fpr_cat, tpr_cat, 
    color='red', lw=2.5,
    label=f'CatBoost (AUC = {roc_auc_cat:.4f})'
)

# Plot XGBoost
plt.plot(
    fpr_xgb, tpr_xgb, 
    color='blue', lw=2.5,
    label=f'XGBoost (AUC = {roc_auc_xgb:.4f})'
)

# Plot garis diagonal (random classifier)
plt.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Random Classifier')

# Konfigurasi plot
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR) / Recall', fontsize=12)
plt.title('Kurva ROC: Komparasi Kinerja XGBoost vs CatBoost (SMOTE)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Interpretasi AUC-ROC:")
print(f"   - XGBoost: {roc_auc_xgb:.4f}")
print(f"   - CatBoost: {roc_auc_cat:.4f}")
if roc_auc_xgb > roc_auc_cat:
    print(f"   ✅ XGBoost memiliki performa lebih baik (+{(roc_auc_xgb-roc_auc_cat):.4f})")
else:
    print(f"   ✅ CatBoost memiliki performa lebih baik (+{(roc_auc_cat-roc_auc_xgb):.4f})")

---

## Kesimpulan

Dari hasil evaluasi di atas, dapat disimpulkan:

1. **Dataset**: Bank Marketing Dataset dengan 41.188 sampel dan rasio ketidakseimbangan kelas sebesar ~7.88:1 (no:yes)

2. **Teknik Penanganan Data Tidak Seimbang**: SMOTE (Synthetic Minority Over-sampling Technique) berhasil menyeimbangkan distribusi kelas pada data training

3. **Performa Model**:
   - Kedua model menunjukkan performa yang baik setelah penerapan SMOTE
   - Metrik evaluasi mencakup: Accuracy, Precision, Recall, F1-Score, dan AUC-ROC
   - Kurva ROC menunjukkan kemampuan model dalam membedakan kelas positif dan negatif

4. **Rekomendasi**: Model dengan nilai AUC-ROC tertinggi lebih direkomendasikan untuk kasus klasifikasi data tidak seimbang ini